# **TAS (Tele Assistance System) Dimensional Analysis Modelling**
NOTE: _[DASA CASE STUDY 1]_

## **Summary**

This notebook is focused on three main objectives:
1. summarizing the key aspects of the Tele Assistance System (TAS) architecture and its adaptive capabilities in the context of telehealth services for chronic patients.
2. Modelling the TAS architecture using appropriate design notations and tools to visualize its components and interactions.
3. Simulating the TAS behavior under different scenarios to evaluate its performance and adaptability with Queue Network (QN) models.

The results will be used to evaluate the Dimensional Analysis for CUSTOM Architecture (DASA) methodology, its CUSTOM tool (PyDASA) and its effectiveness in modelling and Quality Scenarios (QS) trade-off in self-adaptive-systems (SAS).

---

## **Software Architecture**
- TAS (Tele Assistance System) operates in a dynamic environment where service quality, availability, and user needs frequently change.
- The TAS is further subdivided into Controller and Target System subsystem components.
- The Controller is responsible for managing the overall system behavior, while the Target System focuses on executing specific tasks related to patient care.
- The TAS target systems follows a Service-oriented architecture (SOA) pattern.
- The TAS Controller follows a MAPE-K (Monitor-Analyze-Plan-Execute-Knowledge) feedback loop for self-adaptation.
- Adaptations focus on maintaining **reliability**, **performance**, and **compliance** with patient care standards (5 specific scenarios).
- ActivFORMS provides the runtime framework for model-based adaptation using runtime models, simulations, and verified decision-making.

---

_**NOTE: MORE DETAILS ON THE ARCHITECTURE IN THE ANALYTICAL MODELLING NOTEBOOK!.**_

---

## **Code**

_**SUMMARY:**_

This code is for the analytical solution of the Case Study (TAS) Dimensional Analysis Model and is structured as follows:
1. Analytical Dimensional Model (DA).
2. Importing necessary libraries and modules.
3. Loading DA default Monte Carlo experimental data.
4. Preparing the DA experimental data.
5. Solving the default DA model numerically (Machine Learning training + testing).
6. Loading DA optimal Monte Carlo experimental data.
7. Preparing the DA optimal experimental data.
8. Solving the DA optimal model numerically (Machine Learning training + testing).
9. Saving the results.
10. Comparing the analytical results (Default Vs. Optimal)
11. Visualizing the results.
12. Generating a summary report.


## **Target System Queue Network Model**

<svg viewBox="0 0 4650 2000" width="1400" height="650">
    <!-- SVG content -->
    <image href="assets/cs1/img/04A - Queue Network.svg" alt="queue-net-diagram" />
    <div align="center"><em>Image 4. TAS Queue Network Diagram.</em></div>
</svg>

### **Necessary Imports**

In [1]:
# -*- coding: utf-8 -*-
# Native imports
import os
import re
import sys
import time
from typing import Union

# Third-party imports, data and numeric
import numpy as np
import pandas as pd

# Machine Learning libraries
import tensorflow as tf
# kera + tensorflow backend
from keras import Sequential
from keras.layers import Dense, Dropout, Input
from keras.optimizers import Adam
# sklearn libraries
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler



# Model Aprox + Visualizartion libraries
from scipy.interpolate import Rbf
from scipy.interpolate import griddata, LinearNDInterpolator
# from statsmodels.nonparametric.smoothers_lowess import lowess
from sklearn.ensemble import GradientBoostingRegressor

import matplotlib.pyplot as plt

# import queue network + models packages
from src.model.queueing import Queue
from src.model.analytical import solve_jackson_network, calculate_net_metrics


# import plot functions + grahics
from src.view.plots import plot_queue_network
from src.view.plots import plot_net_comparison
from src.view.plots import plot_net_difference
from src.view.plots import plot_nodes_heatmap
from src.view.plots import plot_nodes_diffmap
from src.view.plots import plot_performance_coef_chart
from src.view.plots import plot_experiment_coef_chart

### **Function Definitions**

In [2]:
# Simple formatter for console output

def fmt(val: Union[int, float, np.number]) -> Union[str, np.ndarray]:
    """Format a number to 4 decimal places for console output.

    Args:
        val (Union[int, float, np.number, np.ndarray]): The value to format.

    Returns:
        Union[str, np.ndarray]: The formatted value as a string or an array of strings.
    """
    if isinstance(val, (int, float, np.number)):
        if np.isnan(val) or np.isinf(val):
            return str(val)
        return f"{val:.4f}"
    elif isinstance(val, np.ndarray):
        return np.array([fmt(x) for x in val])
    return val

In [3]:
# Load configuration from a CSV file
def load(path: str, fname: str) -> pd.DataFrame:
    """Load configuration from a CSV file.

    Args:
        path (str): The directory path where the CSV file is located.
        fname (str): The name of the CSV file to load.

    Returns:
        pd.DataFrame: A DataFrame containing the configuration data.
            CSV format:
                - node: <node_id>
                - miu: <mean_service_time>
                - c: <service_channels>
                - K: <buffer_capacity | max_queue_length>
                - lambda0: <initial_arrival_rate>
                - L0: <initial_queue_length>
                - pm: <matrix_routing_probabilities>
    """
    # path = os.path.dirname(__file__)
    _file_path = os.path.join(path, fname)
    print(f"Loading configuration from: {_file_path}")
    df = pd.read_csv(_file_path)
    return df

In [4]:
# save dataframes in CSV files
def save(path: str, fname: str, data: pd.DataFrame) -> None:
    """Save a DataFrame to a CSV file.

    Args:
        path (str): The directory path where the CSV file will be saved.
        fname (str): The name of the CSV file to save.
        data (pd.DataFrame): The DataFrame containing the data to save.
    """
    # path = os.path.dirname(__file__)
    _file_path = os.path.join(path, fname)
    print(f"Saving data to: {_file_path}")
    data.to_csv(_file_path, index=False)

In [5]:
# calculate queue length, with ququeing theory formulas
def queue_length(lambda_z: float,
                 mu: float,
                 c: int,
                 K: int) -> float:
    """Calculate the average queue length (Lq) for an M/M/c/K queueing system.

    Args:
        lambda_z (float): The arrival rate (λ).
        mu (float): The service rate (μ).
        c (int): The number of service channels.
        K (int): The maximum number of customers in the system.

    Returns:
        float: The average number of requests in the system (L, Lq).
    """
    adjusted = False
    # to avoid unstable system
    if lambda_z > mu * c:
        lambda_z = mu * 0.9999
        adjusted = True

    # create queue object
    q = Queue("M/M/s/K", lambda_z, mu, c, K)
    # calculate queue metrics
    q.calculate_metrics()
    ans = q.avg_len

    # to know afterwars what data is adjusted in df
    if adjusted is True:
        # ans = -1 * ans
        ans = (lambda_z, mu, c, K, ans)

    # check the results in the dataframe afterwars
    return ans

In [6]:
# calculate queue time, with Little's Law
def queue_time(lambda_z: float, len_queue: float) -> float:
    """Calculate the average time in the queue system (W, Wq).

    Args:
        len_queue (float): The average queue length (L, Lq).
        lambda_z (float): The arrival rate (λ).

    Returns:
        float: The average wait time in the queue (W, Wq).
    """
    if lambda_z == 0:
        return float("inf")  # Avoid division by zero
    wait_time = len_queue / lambda_z
    return wait_time

In [7]:
def adjust_rate(rate:float, err:float=0.0) -> float:
    """Adjust the arrival rate by a given error percentage.

    Args:
        rate (float): The original arrival rate (λ).
        err (float): The error percentage to adjust the rate by.

    Returns:
        float: The adjusted arrival rate otherwise known as departure rate (x).
    """
    adjusted_rate = rate * (1 + err)
    return adjusted_rate

In [8]:
def uniform_int(low: int, high: int) -> int:
    """Generate a random integer within a specified range.

    Args:
        low (int): The lower bound of the range (inclusive).
        high (int): The upper bound of the range (exclusive).

    Returns:
        int: A random integer between low and high.
    """
    return np.random.randint(low, high)

In [9]:
def shifted_uniform_int(low: int, high: int, shift: int) -> int:
    """Generate a random integer within a specified range and apply a shift.

    Args:
        low (int): The lower bound of the range (inclusive).
        high (int): The upper bound of the range (exclusive).
        shift (int): The value to shift the generated number by.

    Returns:
        int: A random integer between low and high, shifted by the specified amount.
    """
    return np.random.randint(low, high) + shift

In [10]:
def uniform(low: float, high: float) -> float:
    """Generate a random float within a specified range.

    Args:
        low (float): The lower bound of the range (inclusive).
        high (float): The upper bound of the range (exclusive).

    Returns:
        float: A random float between low and high.
    """
    return np.random.uniform(low, high)

In [11]:
# path = os.path.dirname(__file__)\
PATH = os.getcwd()
print(f"Notebook path: {PATH}")

Notebook path: c:\Users\Felipe\OneDrive\Documents\GitHub\DASA-Design\PyDASA-Case-Studies


In [12]:
# Folder names
asset_folder = "assets"
docs_folder = "docs"
img_folder = "img"
data_folder = "data"
report_folder = "reports"
results_folder = "results"
cs_folder = "cs1"

# defining the contour lines for plots
contour_vals = [
    0.05,   # 0
    0.1,    # 1
    0.2,    # 2
    0.3,    # 3
    0.4,    # 4
    0.5,    # 5
    0.6,    # 6
    # 0.65,   # 7
    0.7,    # 8
    0.8,    # 9
    0.9,    # 10
    0.95,   # 11
    0.99,   # 12
    # 1.0,    # 13
    # 1.2,    # 14
    # 1.5,    # 15
    # 2.0,    # 16
    # 3.0,    # 17
]

In [21]:
# setting case study data folder
file_path = os.path.join(PATH, results_folder, cs_folder, data_folder)
print(f"Data path: {file_path}")

Data path: c:\Users\Felipe\OneDrive\Documents\GitHub\DASA-Design\PyDASA-Case-Studies\results\cs1\data


In [28]:

dflt_da_exp = load(file_path, "dflt_dimensional_net_coeffs.csv")
dflt_da_exp.head()
dflt_da_exp.info()

Loading configuration from: c:\Users\Felipe\OneDrive\Documents\GitHub\DASA-Design\PyDASA-Case-Studies\results\cs1\data\dflt_dimensional_net_coeffs.csv
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 130000 entries, 0 to 129999
Data columns (total 8 columns):
 #   Column                                         Non-Null Count   Dtype  
---  ------                                         --------------   -----  
 0   \Pi_{1}=\frac{\lambda_{TAS}*W_{TAS}}{c_{TAS}}  130000 non-null  float64
 1   \Pi_{2}=\frac{\mu_{TAS}}{\lambda_{TAS}}        130000 non-null  float64
 2   \Pi_{3}=\frac{\chi_{TAS}}{\lambda_{TAS}}       130000 non-null  float64
 3   \Pi_{4}=\frac{K_{TAS}}{c_{TAS}}                130000 non-null  float64
 4   \Pi_{5}=\frac{L_{TAS}}{c_{TAS}}                130000 non-null  float64
 5   \eta=\Pi_{Eff}=\frac{\chi}{\mu}                130000 non-null  float64
 6   S=\Pi_{Stall}=\ln(\frac{\lambda*W}{K} + 1)     130000 non-null  float64
 7   O=\Pi_{Occ}=\frac{L}{K}                  

In [29]:
print("--- Extracting Coefficients Names ---")
coef_cols = dflt_da_exp.columns.tolist()
occ_coef = coef_cols[-1]
stall_coef = coef_cols[-2]
eff_coef = coef_cols[-3]
print(f"Occupancy Coefficient: {occ_coef}")
print(f"Stall Coefficient: {stall_coef}")
print(f"Efficiency Coefficient: {eff_coef}")

--- Extracting Coefficients Names ---
Occupancy Coefficient: O=\Pi_{Occ}=\frac{L}{K}
Stall Coefficient: S=\Pi_{Stall}=\ln(\frac{\lambda*W}{K} + 1)
Efficiency Coefficient: \eta=\Pi_{Eff}=\frac{\chi}{\mu}


In [31]:
print("--- Cleaning Invalid Values from Plot Data ---")
# clear plot data
# clear x-axis: occ <= 1
LIM_FLOW = 1.0
LIM_BUFFER = 1.0
print(f"cleaning invalid values in: {eff_coef}")
dflt_da_exp = dflt_da_exp[dflt_da_exp[eff_coef] <= LIM_FLOW]
print(f"Plot data with {eff_coef} < {LIM_FLOW}: {dflt_da_exp.shape}")

# # clear y-axis: stall <= K/c <= something when K is max and c is min
# print(f"cleaning invalid values in: {stall_coef}")
# dflt_da_exp = dflt_da_exp[dflt_da_exp[stall_coef] <= (K_MAX / C_MIN)]
# print(f"Plot data with {stall_coef} < {K_MAX}/{C_MIN}: {dflt_da_exp.shape}")

# clear contour/z-axis: eta <= 1
print(f"cleaning invalid values in: {occ_coef}")
dflt_da_exp = dflt_da_exp[dflt_da_exp[occ_coef] <= LIM_BUFFER]
print(f"Plot data with {occ_coef} < {LIM_BUFFER}: {dflt_da_exp.shape}")
dflt_da_exp.info()

--- Cleaning Invalid Values from Plot Data ---
cleaning invalid values in: \eta=\Pi_{Eff}=\frac{\chi}{\mu}
Plot data with \eta=\Pi_{Eff}=\frac{\chi}{\mu} < 1.0: (96668, 8)
cleaning invalid values in: O=\Pi_{Occ}=\frac{L}{K}
Plot data with O=\Pi_{Occ}=\frac{L}{K} < 1.0: (96668, 8)
<class 'pandas.core.frame.DataFrame'>
Index: 96668 entries, 0 to 129999
Data columns (total 8 columns):
 #   Column                                         Non-Null Count  Dtype  
---  ------                                         --------------  -----  
 0   \Pi_{1}=\frac{\lambda_{TAS}*W_{TAS}}{c_{TAS}}  96668 non-null  float64
 1   \Pi_{2}=\frac{\mu_{TAS}}{\lambda_{TAS}}        96668 non-null  float64
 2   \Pi_{3}=\frac{\chi_{TAS}}{\lambda_{TAS}}       96668 non-null  float64
 3   \Pi_{4}=\frac{K_{TAS}}{c_{TAS}}                96668 non-null  float64
 4   \Pi_{5}=\frac{L_{TAS}}{c_{TAS}}                96668 non-null  float64
 5   \eta=\Pi_{Eff}=\frac{\chi}{\mu}                96668 non-null  float64
 6   S=

### **Dimensional Model**

#### **Defining the Controller**

In [ ]:
data = dflt_da_exp.copy()

# Extract X (features) and y (target)
# Using Occupation and Efficiency as inputs
X = data[[occ_coef, eff_coef]].values
y = data[stall_coef].values  # Predicting Stall coefficient

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

# y_log = np.log1p(y)  # log(1+y) to handle zeros

# # 2. Train-test split on transformed data
# X_train, X_test, y_log_train, y_log_test = train_test_split(
#     X, y, test_size=0.2, random_state=42)

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
y_train_scaled = scaler.fit_transform(y_train.reshape(-1, 1)).flatten()
y_test_scaled = scaler.transform(y_test.reshape(-1, 1)).flatten()

# Build a simple neural network
model = Sequential([
    Input(shape=(X_train.shape[1],)),
    Dense(4*64, activation="relu"),
    Dropout(0.25),  # Add dropout for regularization
    Dense(4*32, activation="relu"),
    Dropout(0.25),
    Dense(4*16, activation="relu"),
    Dense(1)
])

### **Base Configuration**

#### **Loading Data**

#### **Preprocessing Data**

##### **Training Phase**

###### **Plotting Results**

##### **Testing Phase**

###### **Plotting Results**

### **Optimized Configuration**

## **Results**

### **Compare Results**

### **Saving Results**

## **Analysis**

### **Graph Analysis**

## **Conclusion**

Understanding Contour Behavior in Your Queue System Chart
The contour lines in your plot represent queue length values (10, 20, 30, 40, 50), and they appear higher and closer to the Y-axis for higher values due to several important queue theory principles:

Why This Pattern Occurs
Exponential Queue Growth: In queueing theory, as system utilization (ρ = λ/μ) approaches 1.0, queue length grows exponentially rather than linearly. This fundamental property creates the compressed contour pattern you're seeing.

Dimensionless Variables Relationship: Your dimensionless plot shows that:

Small changes in the X-axis variable (first Pi coefficient) cause large changes in queue length when the system is near capacity
The Y-axis variable (second Pi coefficient) has less impact on queue length at higher utilization levels
Log-Log Scale Effect: You're using logarithmic scales on both axes, which compresses the higher values and spreads out the lower values, making this pattern more pronounced.

Queue Theory Interpretation
This pattern visualizes a critical queueing system concept: the performance cliff effect. As your system approaches saturation:

Small increases in load (moving right on X-axis) cause dramatically larger queue lengths
Improving service rate (moving up on Y-axis) provides diminishing returns once the system is near saturation
This is why the higher contour lines (40, 50) are compressed toward the right side of the chart and appear to "stack up" near the Y-axis.

Engineering Significance
This pattern in your data suggests:

Your system has a critical utilization threshold beyond which performance degrades rapidly
The "High Utilization Region" you've marked represents the danger zone where small load increases cause large queue growth
The system should be operated with sufficient margin from this threshold to maintain stable performance
This is exactly the kind of relationship a Moody-style chart should reveal - helping identify safe operating regions and performance boundaries in your system.

## **Future Work**

## **References & Sources**
<!-- TODO fix the references, links and details -->
1. [Queueing Theory](https://en.wikipedia.org/wiki/Queueing_theory)
2. [Dimensional Analysis](https://en.wikipedia.org/wiki/Dimensional_analysis)
3. [Simulation in Healthcare](https://www.ncbi.nlm.nih.gov/pmc/articles/PMC6466220/)

---

# **HASTA AKI!!!**